In [1]:
import os
import requests
import json
from dotenv import load_dotenv
from datetime import datetime, timezone
import time

# Define start and end date
def to_unix_mmddyy(date_str: str) -> int:
    """
    Convert a date in U.S. **M/D/YY** format (e.g. '1/1/25') to a Unix timestamp.
    Interprets two‑digit years as 2000‑2099.

    >>> to_unix_mmddyy("1/1/25")
    1735689600
    """
    # Parse with strptime, then convert the naive datetime to UTC
    dt = datetime.strptime(date_str, "%m/%d/%y").replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

start_date = int(datetime.strptime('2025-01-01', '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp())
end_date = int(datetime.now(tz=None).timestamp())
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

# Load environment variables
load_dotenv("API_KEYS.env")
coingecko_api_key = os.getenv("COINGECKO_API_KEY")

# Fetches and saves down a list of asset IDs from Coingecko
def fetch_and_save_asset_list(output_filename):
    if not coingecko_api_key:
        print("Error: API key not found. Please set COINGECKO_API_KEY in the .env file.")
        return

    url = "https://api.coingecko.com/api/v3/coins/list?include_platform=true"
    headers = {
    "accept": "application/json",
    "x-cg-demo-api-key": coingecko_api_key
    }

    try:
        # Make API request
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raises HTTP errors if they occur
        
        # Parse response JSON
        data = response.json()
        
        # Save JSON response to file
        with open(output_filename, "w") as json_file:
            json.dump(data, json_file, indent=4)
        
        print(f"Data successfully saved to {output_filename}")

    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except requests.exceptions.RequestException as req_err:
        print(f"Request error: {req_err}")
    except json.JSONDecodeError:
        print("Error decoding JSON response")
    except Exception as e:
        print(f"Unexpected error: {e}")


    

Start date: 1735689600
End date: 1752634628


In [2]:
# Pull down coin list from Coingecko
output_filename = "coingecko-raw-data/coingecko_asset-list.json"

fetch_and_save_asset_list(output_filename)
print("Done")

Data successfully saved to coingecko-raw-data/coingecko_asset-list.json
Done


In [3]:
# Function to save down list of all token IDs on Hyperliquid

def identify_token_ids(file_path, network):
    with open(file_path, 'r') as f:
        data = json.load(f)
        
    filtered = []
    for entry in data:
        token_id = entry.get('id', [])
        platform_dict = entry.get('platforms', [])
        if network not in platform_dict:
            continue
        
        else:
            filtered.append(token_id)

    print(filtered)
    
    return filtered
            

In [4]:
# Call the function

file_path = output_filename
network = 'hyperliquid'

coins = identify_token_ids(file_path, network)

['alright-buddy', 'atehun', 'autist', 'cat-2', 'catbal', 'cz-on-hyperliquid', 'depin', 'ethena-usde', 'farm-2', 'felix-feusd', 'fractrade', 'h', 'hedgewaterdao', 'hyena', 'hyperfly', 'hyperlauncher', 'hyperliquid', 'hyper-usd', 'hypurr-fun', 'jeff-3', 'last-usd', 'liquidlaunch', 'liquina', 'omnix', 'ora-coin', 'pip-3', 'pump-fun', 'purr-2', 'rugman', 'sentiient', 'sovrun', 'tether-gold-tokens', 'unit-bitcoin', 'unit-ethereum', 'unit-fartcoin', 'unit-solana', 'usdt0', 'vapor', 'vault-overflow', 'vegas', 'yeeti']


In [5]:
# Function to get and save down coin data
def fetch_and_save_historical_chart_data(coins, start, end, t):
    if not coingecko_api_key:
        print("Error: API key not found. Please set COINGECKO_API_KEY in the .env file.")
        return

    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart/range?vs_currency=usd&from={start}&to={end}"
    headers = {
        "accept": "application/json",
        "x-demo-pro-api-key": coingecko_api_key
    }
        
    print("Requesting:", url)

    try:
        with requests.get(url, headers=headers, timeout=t) as resp:
            resp.raise_for_status() # raise if 4xx/5xx
            data = resp.json() # read the body
        
        # Save it to a file
        output_filename = f'coingecko-raw-data/coingecko_{coin}_spot_strict_list_data.json'
        with open(output_filename, "w") as f:
            json.dump(data, f, indent=2)
        
        print(f"Data saved to {output_filename}")
    except:
        print(f'Error saving down data for {coin}')

In [6]:
# Pull down historical coin data for the defined list

t = 30
for coin in coins:
    time.sleep(t)
    fetch_and_save_historical_chart_data(
        coin,
        start_date,
        end_date,
        t
    )
print("Done")

Requesting: https://api.coingecko.com/api/v3/coins/alright-buddy/market_chart/range?vs_currency=usd&from=1735689600&to=1752634628
Data saved to coingecko-raw-data/coingecko_alright-buddy_spot_strict_list_data.json
Requesting: https://api.coingecko.com/api/v3/coins/atehun/market_chart/range?vs_currency=usd&from=1735689600&to=1752634628
Data saved to coingecko-raw-data/coingecko_atehun_spot_strict_list_data.json
Requesting: https://api.coingecko.com/api/v3/coins/autist/market_chart/range?vs_currency=usd&from=1735689600&to=1752634628
Data saved to coingecko-raw-data/coingecko_autist_spot_strict_list_data.json
Requesting: https://api.coingecko.com/api/v3/coins/cat-2/market_chart/range?vs_currency=usd&from=1735689600&to=1752634628
Data saved to coingecko-raw-data/coingecko_cat-2_spot_strict_list_data.json
Requesting: https://api.coingecko.com/api/v3/coins/catbal/market_chart/range?vs_currency=usd&from=1735689600&to=1752634628
Data saved to coingecko-raw-data/coingecko_catbal_spot_strict_lis